# Inspect Forte visual search

Diagnostic notebook for the packet behind `visual_search_forte.html`.

It **loads the saved orchard** under `artifacts/forte_blogs/` and walks each
construction step. It does **not** retrain ModernBERT, rebuild MiniLM
embeddings, or call OpenAI. Those live in `scripts/export_forte_orchard.py`
and `scripts/adapt_forte_to_mockup.py`.

Use this to answer two practical questions:

1. **Which tree to browse** — semantic (topic geometry) vs CODE (process-stage geometry, fused with the same embeddings).
2. **Which cuts to trust / recut** — Calinski top partitions, `max_depth` lumps, singleton outliers, and repeated LLM labels.

Run top to bottom from the `orchard-view/` directory. Change `TREE_ID` in the
setup cell to switch the tree under inspection.


## 0. Pipeline map

```text
279 markdown posts
        │
        ├─► thin markdown loader (frontmatter → title/metadata, body → text, stem → item_id)
        │
        ├─► CODE YAML seed
        │         │
        │         ├─► orchard_taxonomy_definition_v1
        │         ├─► 20 synthetic chunks + ~25 heading-explicit sections  (training only)
        │         └─► ModernBERT + logistic head  (NOT TaxonomyModel.fit)
        │                   │
        │                   └─► transform() 279 posts → CODE distributions
        │
        └─► OrchardBuilder
                  taxonomies=[CODE]
                  include_semantic_with_taxonomies=True
                  explicit CODE fusion weights
                          │
                          ├─► shared layer matrices
                          │     MiniLM cosine, TF-IDF cosine, CODE Jensen–Shannon
                          │
                          ├─► semantic tree   MiniLM 0.66 + TF-IDF 0.34
                          └─► CODE tree       CODE_js 0.50 + MiniLM 0.30 + TF-IDF 0.20
                                    │
                                    ├─► Calinski-optimal dynamic cut on fused D (the same D that built Z)
                                    ├─► gpt-4.1-mini labels on cut-visible internals
                                    └─► adapt: semantic → HTML "domain" slot, CODE → HTML "function" slot
                                              visual_search_forte.html + standalone folder
```

Each section below is one of those boxes, with the **choice** that was locked in
and a diagnostic you can rerun on the saved artifacts.


## Setup — load the saved packet

`TREE_ID` is `"semantic"` or `"CODE"`. Later cells that say "this tree" follow it.
Both trees are loaded; comparison cells use both.


In [1]:
from __future__ import annotations

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import yaml
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

from orchard import Orchard, build_dynamic_cut, validate_dynamic_cut
from orchard.backends.fusion import fuse_to_dissimilarity
from orchard.cuts import get_optimal_cut_labels
from orchard.tree import default_node_label

VIEW = Path.cwd() if (Path.cwd() / "artifacts" / "forte_blogs").exists() else Path.cwd() / "orchard-view"
sys.path.insert(0, str(VIEW / "scripts"))
from export_forte_orchard import (  # noqa: E402
    CODE_FUSION_WEIGHTS,
    CUT_PARAMS,
    EXPECTED_DOCUMENT_COUNT,
    GOLD_STEM,
    POSTS_DIR,
    SEMANTIC_FUSION_WEIGHTS,
    SEED_YAML,
    align_square_to_item_ids,
    collect_training,
    load_posts,
    yaml_to_definition,
)

PACKET = VIEW / "artifacts" / "forte_blogs"
TREE_ID = "semantic"  # or "CODE"

orchard = Orchard.load(PACKET / "orchard")
cuts = {
    name: json.loads((PACKET / "cuts" / f"{name}.calinski_optimal.json").read_text(encoding="utf-8"))
    for name in ("semantic", "CODE")
}
builder_params = json.loads((PACKET / "builder_params.json").read_text(encoding="utf-8"))
node_descriptions = json.loads((PACKET / "node_descriptions.json").read_text(encoding="utf-8"))
train_manifest = json.loads((PACKET / "training" / "manifest.json").read_text(encoding="utf-8"))
code_definition = json.loads((PACKET / "code_taxonomy.json").read_text(encoding="utf-8"))

tree = orchard.tree(TREE_ID)
cut = cuts[TREE_ID]
profiles = orchard.metadata["profiles"]

print("docs", len(orchard.documents), "expected", EXPECTED_DOCUMENT_COUNT)
print("trees", list(orchard.trees))
print("layers", sorted(orchard.layer_matrices))
print("inspecting", TREE_ID, "leaves", tree.leaf_count, "cut k", cut["top_partition_count"])


docs 279 expected 279
trees ['CODE', 'semantic']
layers ['CODE_raw_js', 'description_minilm_centered_cosine', 'tfidf_cosine']
inspecting semantic leaves 279 cut k 3


## 1. Corpus — 279 Forte posts as orchard `Document`s

**Choice.** Orchard’s directory loader only accepts `*.txt` and does not strip
YAML frontmatter, so orchard-view owns a thin markdown loader. The scraper
corpus is read-only.

| Field | Source |
|---|---|
| `item_id` | markdown filename stem |
| `title` | frontmatter `title` (fallback: stem) |
| `text` | body after `---` |
| `metadata` | remaining frontmatter |
| `source` | `"forte_blogs"` |

Training chunks from the CODE YAML are **not** in this corpus. They never
become orchard leaves or HTML documents.


In [2]:
documents = load_posts(POSTS_DIR)
assert len(documents) == len(orchard.documents) == EXPECTED_DOCUMENT_COUNT
assert {d.item_id for d in documents} == {d.item_id for d in orchard.documents}

doc = orchard.documents[0]
print("sample", doc.item_id)
print("title ", doc.title)
print("chars ", len(doc.text))
print("keys  ", sorted((doc.metadata or {}).keys())[:12])
print()
print(doc.text[:400].replace("\n", " "))


sample 0001-the-ai-second-brain-is-back-what-8217-s-new-for-cohort-2
title  The AI Second Brain Is Back: What&#8217;s New for Cohort 2
chars  14882
keys   ['archive_page', 'archive_position', 'archive_url', 'author', 'content_note', 'published', 'retrieved_at', 'source_category', 'source_site', 'source_type', 'source_url', 'title']

# The AI Second Brain Is Back: What&#8217;s New for Cohort 2  In March, we launched the [inaugural cohort](https://fortelabs.com/blog/introducing-the-ai-second-brain/) of The AI Second Brain (AISB), a new immersive education program years in the making.  It marked my return to teaching after three years of intensive research and experimentation with AI. I felt like an explorer returning from the w


## 2. CODE taxonomy definition

**Choice.** AppWorld used Domain + Function. Forte has no app-family key and
the corpus is one author’s knowledge-management blog, so the second tree is
Tiago Forte’s CODE stages: Capture / Organize / Distill / Express.

The YAML seed is converted to `orchard_taxonomy_definition_v1`. Cues are
harvested from each label’s “typical actions include …” phrase — they exist
for the definition, but the **shipped classifier does not use cue/TF-IDF**.


In [3]:
seed = yaml.safe_load(SEED_YAML.read_text(encoding="utf-8"))
converted = yaml_to_definition(seed)
assert converted["label_order"] == code_definition["label_order"] == ["capture", "organize", "distill", "express"]

print("name     ", code_definition["name"])
print("structure", code_definition["structure"])
print("transform shipped on orchard:", orchard.metadata.get("taxonomy_transform"))
print()
for row in code_definition["labels"]:
    print(f"{row['label_id']:10}  cues={row['cues']}")
    print(f"{'':10}  {row['definition'][:140].strip()}…")
    print()


name      CODE
structure flat
transform shipped on orchard: modernbert_logistic

capture     cues=['save', 'clip', 'record', 'note', 'photograph', 'bookmark', 'jot', 'down']
            Capture is the selective intake of information judged worth retaining. It includes noticing, collecting, recording, clipping, saving, or oth…

organize    cues=['file', 'move', 'group', 'associate', 'tag', 'link', 'sort', 'assign']
            Organize is the placement of captured information into useful context so it can be found and applied when needed. It emphasizes actionabilit…

distill     cues=['highlight', 'summarize', 'extract', 'condense', 'emphasize', 'synthesize', 'rank']
            Distill is the reduction of retained information to its most relevant, useful, or essential elements. It includes highlighting, summarizing,…

express     cues=['write', 'publish', 'present', 'teach', 'build', 'decide', 'send', 'create', 'ship']
            Express is the conversion of accumulated knowledge into

## 3. CODE training harvest — sections, not whole posts

**Choice.** A whole blog post usually mentions several CODE stages. Training
on full posts would smear the four classes. The harvest is:

1. **20 synthetic YAML `training_chunks`** (5 per class), ids `seed.{label}.{i}`.
2. **Heading-explicit real sections** — H1–H3 whose heading contains **exactly
   one** of `{capture, organize, distill, express}`. Mixed headings such as
   “Organize and Distill” are skipped.
3. Gold check: post `0027-building-a-second-brain-the-definitive-introductory-guide`
   must contribute all four stages.

These 45 items train the head only. `OrchardBuilder.build()` sees the 279 posts.


In [4]:
train_docs, train_labels, harvested = collect_training(seed, POSTS_DIR)
items = train_manifest["items"]
by_source = Counter(row["source"] for row in items)
by_label = Counter(row["label"] for row in items)
gold = [row for row in items if row["path"] and GOLD_STEM in row["path"]]

print("manifest items", len(items), "harvested now", len(harvested))
print("by source", dict(by_source))
print("by label ", dict(by_label))
print("gold headings")
for row in gold:
    print(f"  {row['label']:10}  {row['heading']}")
print()
print("training ids are orchard leaves?", any(i.startswith("seed.") for i in tree.item_ids))


manifest items 45 harvested now 45
by source {'yaml_chunk': 20, 'heading_section': 25}
by label  {'capture': 18, 'organize': 14, 'distill': 7, 'express': 6}
gold headings
  capture     Capture only the most important information
  capture     Utilize capture tools
  organize    Organize for actionability
  distill     Distill opportunistically, a little bit at a time
  express     Express your unique ideas and experiences

training ids are orchard leaves? False


## 4. Neural CODE head — why not `TaxonomyModel.fit()`

**Choice.** In the current orchard library, `TaxonomyModel.fit()` still trains
TF-IDF + logistic and sets `taxonomy_transform="cue"`. Calling it would wipe a
neural head. The export script therefore:

```text
TaxonomyModel.from_definition(CODE)
  → ModernBERTFeatureBackend.encode(title + text)
  → LogisticRegression(**HEAD_HYPERPARAMS)     # C=0.1, balanced, max_iter=1000, seed 20260725
  → attach classifier + encoder; vectorizer = None
  → taxonomy_transform = "modernbert_logistic"
  → save_head(artifacts/forte_blogs/code_head.npz)
  → transform() the 279 posts  (not the 45 training items)
```

MiniLM is the **description embedding** for the semantic layer. ModernBERT is
the **taxonomy encoder** only. They are not swapped.

This notebook does not re-encode. Per-document CODE probabilities are not
persisted — only the pairwise `CODE_raw_js` layer. Optional cell at the bottom
re-runs `transform()` if you need class histograms.


In [5]:
print("taxonomy_transform     ", orchard.metadata.get("taxonomy_transform"))
print("taxonomy model         ", builder_params.get("taxonomy_model_id"))
print("head file exists       ", (PACKET / "code_head.npz").is_file())
print("CODE_raw_js persisted  ", "CODE_raw_js" in orchard.layer_matrices)
print("HEAD_HYPERPARAMS note  ", "C=0.1 class_weight=balanced random_state=20260725")


taxonomy_transform      modernbert_logistic
taxonomy model          answerdotai/ModernBERT-base
head file exists        True
CODE_raw_js persisted   True
HEAD_HYPERPARAMS note   C=0.1 class_weight=balanced random_state=20260725


## 5. Builder + fusion — two trees over the same 279 posts

**Choice.** `include_semantic_with_taxonomies=True` so you get a semantic tree
*and* a CODE tree. No Domain / Function taxonomies. No `app_exact_match`
(posts have no complete family key). gpt-4.1-mini is a **labeler**, not an embedder.

Library traps this packet worked around:

| Trap | What we did |
|---|---|
| Packaged fused dicts exist only for trees named `domain` / `function`. A custom `CODE` tree would default to `{CODE_raw_js: 1.0}`. | Pass `taxonomy_weights={"CODE": {…}}` explicitly. |
| Semantic auto-fuses MiniLM 0.66 + TF-IDF 0.34 when `orchard[ml]` is present. | Do **not** inject `TfidfEmbeddingBackend()`. |
| Missing extras would otherwise silently TF-IDF. | `allow_offline_fallback` stays false; extras must be installed. |
| Concatenating heterogeneous feature vectors. | Independent layers, convex weights, `variance_calibrated` fusion. |

Shipped weights (must sum to 1; orchard will not renormalize):

| Tree | Layer | Weight | Role |
|---|---|---|---|
| semantic | `description_minilm_centered_cosine` | 0.66 | what the post is about |
| semantic | `tfidf_cosine` | 0.34 | lexical overlap |
| CODE | `CODE_raw_js` | 0.50 | Jensen–Shannon on CODE distributions |
| CODE | `description_minilm_centered_cosine` | 0.30 | same topic signal as semantic |
| CODE | `tfidf_cosine` | 0.20 | lexical overlap |

CODE is **not** a pure 4-class taxonomy tree. Half of its geometry is the same
semantic/lexical signal as the other tree. That is the main reason the HTML
CODE view does not look like Capture / Organize / Distill / Express folders.


In [6]:
print("embedding_backend   ", builder_params["embedding_backend"])
print("lexical_backend     ", builder_params["lexical_backend"])
print("fusion_mode         ", builder_params["fusion_mode"])
print("offline_fallback    ", builder_params["offline_fallback"])
print("linkage_method      ", builder_params["linkage_method"])
print("layer persist       ", builder_params["layer_matrix_persist"])
print()
for name, profile in profiles.items():
    print(name, dict(profile["weights"]), "mode", profile["fusion_mode"])
print()
print("expected semantic", SEMANTIC_FUSION_WEIGHTS)
print("expected CODE    ", CODE_FUSION_WEIGHTS)
print("active_layers    ", builder_params["active_layers"])


embedding_backend    MiniLMEmbeddingBackend
lexical_backend      TfidfEmbeddingBackend
fusion_mode          variance_calibrated
offline_fallback     False
linkage_method       average
layer persist        compressed

CODE {'CODE_raw_js': 0.5, 'description_minilm_centered_cosine': 0.3, 'tfidf_cosine': 0.2} mode variance_calibrated
semantic {'description_minilm_centered_cosine': 0.66, 'tfidf_cosine': 0.34} mode variance_calibrated

expected semantic {'description_minilm_centered_cosine': 0.66, 'tfidf_cosine': 0.34}
expected CODE     {'CODE_raw_js': 0.5, 'description_minilm_centered_cosine': 0.3, 'tfidf_cosine': 0.2}
active_layers     {'CODE': ['CODE_raw_js', 'description_minilm_centered_cosine', 'tfidf_cosine'], 'semantic': ['description_minilm_centered_cosine', 'tfidf_cosine']}


## 6. Reconstruct the fused D that built each linkage

**Choice.** Cuts must score the **same geometry as Z**. The builder persists
shared `layer_matrices` (`compressed` npz). The export script aligns those
squares to `tree.item_ids` and calls `fuse_to_dissimilarity(...)` with that
tree’s profile weights + `fusion_mode`.

Do **not** pass a fresh TF-IDF matrix or raw CODE probability rows into
Calinski. That would rank a different space than the dendrogram.

`variance_calibrated` returns a dissimilarity with a zero diagonal. That D is
what average-linkage clustered, and what the cutter received as `feature_matrix`.


In [7]:
def fused_d_for(tree_id: str) -> np.ndarray:
    profile = profiles[tree_id]
    source_ids = [doc.item_id for doc in orchard.documents]
    aligned = align_square_to_item_ids(
        orchard.layer_matrices, source_ids, list(orchard.tree(tree_id).item_ids)
    )
    missing = [name for name in profile["weights"] if name not in aligned]
    if missing:
        raise SystemExit(f"{tree_id} missing layers {missing}")
    matrices = {name: aligned[name] for name in profile["weights"]}
    return fuse_to_dissimilarity(
        matrices,
        profile["weights"],
        fusion_mode=profile["fusion_mode"],
    )


def layer_summary(name: str, matrix: np.ndarray) -> None:
    diag = np.diag(matrix)
    off = matrix[~np.eye(matrix.shape[0], dtype=bool)]
    print(
        f"{name:42} shape={matrix.shape}  diag[{diag.min():.4g},{diag.max():.4g}]  "
        f"off[{off.min():.4g},{off.max():.4g}] mean_off={off.mean():.4g}"
    )


print("persisted layers (document order)")
for name, matrix in orchard.layer_matrices.items():
    layer_summary(name, matrix)

print()
fused = {}
for name in ("semantic", "CODE"):
    fused[name] = fused_d_for(name)
    layer_summary(f"fused D [{name}]", fused[name])

matrix = fused[TREE_ID]
print()
print(TREE_ID, "fused D is the cut feature_matrix; rows == leaves", matrix.shape[0] == tree.leaf_count)


persisted layers (document order)
CODE_raw_js                                shape=(279, 279)  diag[1,1]  off[0.607,1] mean_off=0.9367
description_minilm_centered_cosine         shape=(279, 279)  diag[1,1]  off[0.278,0.9538] mean_off=0.4987
tfidf_cosine                               shape=(279, 279)  diag[1,1]  off[0.03467,0.8182] mean_off=0.3864

fused D [semantic]                         shape=(279, 279)  diag[0,0]  off[0,6.811] mean_off=4.651
fused D [CODE]                             shape=(279, 279)  diag[0,0]  off[0,6.082] mean_off=2.736

semantic fused D is the cut feature_matrix; rows == leaves True


### Layer agreement — how independent are the signals?

If MiniLM and TF-IDF are nearly the same ranking, fusion is mostly a
reweight. If `CODE_raw_js` disagrees with MiniLM, the CODE tree can still
split process-stage from topic — but only in the 0.50 slice of its weights.


In [8]:
def upper_triangle(matrix: np.ndarray) -> np.ndarray:
    idx = np.triu_indices(matrix.shape[0], k=1)
    return np.asarray(matrix)[idx]


def rankdata(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(values), dtype=np.float64)
    return ranks


def spearman(a: np.ndarray, b: np.ndarray) -> float:
    ra, rb = rankdata(a), rankdata(b)
    ra = ra - ra.mean()
    rb = rb - rb.mean()
    denom = np.sqrt((ra * ra).sum() * (rb * rb).sum())
    return float((ra * rb).sum() / denom) if denom else float("nan")


source_ids = [doc.item_id for doc in orchard.documents]
aligned_all = align_square_to_item_ids(orchard.layer_matrices, source_ids, list(tree.item_ids))
layer_names = list(aligned_all)
print(f"Spearman on upper-triangle pairs ({TREE_ID} leaf order)")
print(f"{'':42}", "  ".join(f"{n[:18]:>18}" for n in layer_names))
for left in layer_names:
    row = [spearman(upper_triangle(aligned_all[left]), upper_triangle(aligned_all[right])) for right in layer_names]
    print(f"{left:42}", "  ".join(f"{v:18.3f}" for v in row))


Spearman on upper-triangle pairs (semantic leaf order)
                                                  CODE_raw_js  description_minilm        tfidf_cosine
CODE_raw_js                                             1.000               0.293               0.299
description_minilm_centered_cosine                      0.293               1.000               0.231
tfidf_cosine                                            0.299               0.231               1.000


## 7. Trees — same leaves, different Z

Both trees cover the same 279 `item_id`s. Canonical node ids are membership
hashes of descendant sets, so the **root id is shared** (same full set).
Internal ids diverge as soon as the two linkages group documents differently.
Average linkage (`method="average"`) on each fused D.

The HTML switcher is not two corpora. It is two partitions of one corpus.


In [9]:
sem = orchard.tree("semantic")
code_tree = orchard.tree("CODE")
print("shared item_ids", list(sem.item_ids) == list(code_tree.item_ids))
print("shared root id ", sem.root_node_id == code_tree.root_node_id)
print("root           ", sem.root_node_id)
print("semantic internals", sum(1 for n in sem.nodes.values() if n["kind"] == "internal"))
print("CODE internals    ", sum(1 for n in code_tree.nodes.values() if n["kind"] == "internal"))
print("shared node ids   ", len(set(sem.nodes) & set(code_tree.nodes)), "(root + any identical memberships)")
print()
print(TREE_ID, "active labels", tree.active_label_set, len(tree.labels[tree.active_label_set]))
print(TREE_ID, "root label  ", tree.label_for(tree.root_node_id))


shared item_ids True
shared root id  True
root            member_9d66c7a645a8127206a5bd2e006df084fad58c5b80c47ffd75d5622b37d07822
semantic internals 278
CODE internals     278
shared node ids    365 (root + any identical memberships)

semantic active labels gpt41mini 344
semantic root label   semantic


## 8. Calinski-optimal dynamic cut

**Choice.** Same parameters as the AppWorld packet. Not a hardcoded
`cluster_count=3`. `top_criterion="optimal"` asks the cutter to pick k by
Calinski–Harabasz **inside** `min_width..max_width` (3..10), then recurse
with the same optimizer down to `max_depth=5`.

| Param | Value | Why it matters |
|---|---|---|
| `top_criterion` | `optimal` | search k, do not pin it |
| `cut_optimizer` | `calinski_harabasz_score` | polarity `+1` (higher is better) |
| `min_width` / `max_width` | 3 / 10 | window; unconstrained CH cannot choose 2 or 11+ |
| `target_width` | `null` | only required for non-optimal criteria |
| `max_depth` | 5 | deeper lumps become `direct_leaf_node_ids` + a warning |
| `threshold_steps` | 32 | candidate distance grid for subcuts |
| `feature_matrix` | fused D | must match Z |

`Orchard.save` does **not** persist cuts. The JSON under `artifacts/forte_blogs/cuts/`
is the view’s source of truth for which internals are visible.

Caveat inherited from AppWorld: sklearn’s Calinski scores rows of `feature_matrix`
as Euclidean feature vectors. Here those rows are rows of a dissimilarity
matrix. That is why using the **same** D as linkage still matters — Z and the
score at least share a geometry — but CH is not scoring the original MiniLM
space.


In [10]:
print("CUT_PARAMS", CUT_PARAMS)
print()
for name in ("semantic", "CODE"):
    payload = cuts[name]
    print(f"=== {name} ===")
    print("  top_criterion", payload["top_criterion"], "optimizer", payload["cut_optimizer"])
    print("  requested k  ", payload["cluster_count_requested"], "resolved k", payload["top_partition_count"])
    print("  width        ", payload["width"])
    print("  max_depth    ", payload["max_depth"])
    print("  warnings     ", len(payload.get("warnings") or []))
    for warning in payload.get("warnings") or []:
        print("   -", warning)
    print()


CUT_PARAMS {'top_criterion': 'optimal', 'cut_optimizer': 'calinski_harabasz_score', 'cut_polarity': 1, 'min_width': 3, 'max_width': 10, 'target_width': None, 'max_depth': 5, 'threshold_steps': 32}

=== semantic ===
  top_criterion optimal optimizer calinski_harabasz_score
  requested k   3 resolved k 3
  width         {'minimum': 3, 'maximum': 10, 'target': None, 'stop': 10}
  max_depth     5
  warnings      4
   - member_d487359888738487bce30a32fb4f59c5cdda6b6b6cad287d9507349f543ad00e retains 18 leaves at max_depth=5
   - member_fa3ae54a7efaa0b80a428bfd5f2a36451c75c6c3cd5433b757456f4e485861ed retains 30 leaves at max_depth=5
   - member_28e166087735dba5cfb683e9f8057ad48baa23bfb6bc07d9e8b94534dcbf755d retains 12 leaves at max_depth=5
   - member_dc514307b8cea06d9fe014713710f3cf3dcff3378361662d681c97fac79dda1f retains 12 leaves at max_depth=5

=== CODE ===
  top_criterion optimal optimizer calinski_harabasz_score
  requested k   3 resolved k 3
  width         {'minimum': 3, 'maximum': 1

### Walk helper

Cut JSON keeps internal `children` **and** `direct_leaf_node_ids` on the same
node. That is the HTML shape: a cluster can contain both subclusters and
posts that did not get a further split (depth cap, or no improving cut).

`walk_cut_json` in orchard promotes those leaves into `children` — useful for
a viewer, wrong as a round-trip artifact. This helper prints the raw cut.


In [11]:
def walk_cut(
    node: dict,
    tree,
    *,
    depth: int = 0,
    max_depth: int = 1,
    show_leaf: bool = False,
    show_kind: bool = False,
) -> None:
    cid = node["canonical_node_id"]
    src = tree.node(cid)
    kind = f"{src['kind']}: " if show_kind else ""
    if src["kind"] == "leaf":
        label = src.get("item_id")
    else:
        label = tree.label_for(cid) or default_node_label(tree, cid)
    n = src["descendant_count"]
    n_hang = len(node.get("direct_leaf_node_ids") or [])
    hang = f"  hang={n_hang}" if n_hang else ""
    print("  " * depth + f"{kind}{str(label or cid)[:80]}  ({n}){hang}")
    if show_leaf:
        for leaf_cid in node.get("direct_leaf_node_ids") or []:
            leaf = tree.node(leaf_cid)
            print("  " * (depth + 1) + f"leaf: {leaf.get('item_id') or leaf_cid}")
    if depth >= max_depth:
        return
    for child in node.get("children") or []:
        walk_cut(
            child,
            tree,
            depth=depth + 1,
            max_depth=max_depth,
            show_leaf=show_leaf,
            show_kind=show_kind,
        )


print(TREE_ID, "shipped cut  k=", cut["top_partition_count"])
walk_cut(cut["root"], tree, max_depth=2, show_leaf=False, show_kind=False)


semantic shipped cut  k= 3
semantic  (279)  hang=1
  Second Brain AI Integration  (204)  hang=1
    Building a Second Brain Updates  (42)
    Second Brain Project Management  (105)
    Global Knowledge Management Expansion  (5)  hang=5
    Digital Note-Taking Strategies  (42)
    PARA Organization Framework  (5)  hang=5
    Second Brain Interviews and History  (4)  hang=4
  Second Brain Foundations  (74)
    Second Brain Knowledge Management  (65)  hang=1
    Information Capture Strategies  (6)  hang=6
    Semantic Knowledge Networks  (3)  hang=3


## 9. Diagnostic — what the top partition actually is

The HTML badge prints `279 docs · N top-level clusters`. N is
`top_partition_count`, which **includes singleton leaves hanging off the root**
(`direct_leaf_node_ids` on the root). A “cluster” of size 1 is an outlier the
cutter promoted to the top, not a browseable folder.

If both trees promote the **same** singleton, that post is an outlier in both
geometries — switching trees will not re-home it.


In [12]:
def partition_rows(tree, cut_payload: dict) -> list[dict]:
    rows = []
    for nid in cut_payload["top_partition_node_ids"]:
        node = tree.node(nid)
        sample = []
        for item_id in node["descendant_item_ids"][:5]:
            doc = tree.documents_by_id.get(item_id)
            sample.append((doc.title if doc and doc.title else item_id)[:70])
        rows.append(
            {
                "id": nid,
                "kind": node["kind"],
                "n": node["descendant_count"],
                "label": tree.label_for(nid) or (node.get("item_id") if node["kind"] == "leaf" else nid[:18]),
                "item_id": node.get("item_id"),
                "sample": sample,
            }
        )
    return rows


print("root direct leaves (true outliers at the top cut)")
for name in ("semantic", "CODE"):
    hanging = cuts[name]["root"].get("direct_leaf_node_ids") or []
    t = orchard.tree(name)
    print(f"  {name}: {len(hanging)}")
    for nid in hanging:
        node = t.node(nid)
        doc = t.documents_by_id.get(node.get("item_id"))
        title = doc.title if doc else node.get("item_id")
        print(f"    {node.get('item_id')}  {title}")

print()
for name in ("semantic", "CODE"):
    print(f"=== {name} top partitions ===")
    for row in partition_rows(orchard.tree(name), cuts[name]):
        print(f"  n={row['n']:3d}  {row['kind']:8}  {row['label']}")
        if row["kind"] == "leaf":
            print(f"           {row['item_id']}")
        else:
            for title in row["sample"]:
                print(f"           · {title}")
    print()


root direct leaves (true outliers at the top cut)
  semantic: 1
    0114-second-brain-case-study-sleep-training-an-infant  Second Brain Case Study: Sleep Training an Infant
  CODE: 1
    0114-second-brain-case-study-sleep-training-an-infant  Second Brain Case Study: Sleep Training an Infant

=== semantic top partitions ===
  n=204  internal  Second Brain AI Integration
           · The AI Second Brain Is Back: What&#8217;s New for Cohort 2
           · Finding Alpha: What&#8217;s Worth Consuming in the Age of AI
           · Is Recall the Second Brain for the AI Era?
           · Introducing the Alumni Mentor Corps
           · Why PARA Is the Key to the AI Era
  n= 74  internal  Second Brain Foundations
           · The 5-Year Journey of Publishing Building a Second Brain
           · Nonprofit Productivity: Leveraging Software to Shape Your Career
           · Bullet Journal: The Difference Between Our First &#038; Second Brain
           · How to Capture Ideas Like a Pro | The Pinkc

## 10. Diagnostic — unconstrained k vs the 3–10 window

The AppWorld domain tree’s live notebook showed unconstrained CH wanting
**k=11** while the dynamic cutter shipped **k=9**, because 11 sits outside
`max_width=10`. Check whether Forte’s shipped k=3 is a window artifact or
what CH actually prefers.

- If unconstrained also picks 3, the window is not the story — the fused
  geometry really has a coarse 3-way split (usually one giant component +
  a smaller one + an outlier).
- If unconstrained wants 8–11, raising `max_width` (or flattening a
  mid-depth cut) is the lever, not a different optimizer.


In [13]:
Z = tree.linkage
print(TREE_ID, "shipped dynamic top_partition_count", cut["top_partition_count"])
print()
for name, metric, polarity in (
    ("silhouette", silhouette_score, 1),
    ("calinski_harabasz", calinski_harabasz_score, 1),
    ("davies_bouldin", davies_bouldin_score, -1),
):
    _, k_flat = get_optimal_cut_labels(
        Z, matrix, metric=metric, polarity=polarity, min_clusters=2, max_clusters=17
    )
    _, k_win = get_optimal_cut_labels(
        Z, matrix, metric=metric, polarity=polarity, min_clusters=3, max_clusters=10
    )
    print(f"{name:22}  unconstrained k={k_flat:2d}  windowed 3..10 k={k_win:2d}")

print()
print("CH score by k (window in [3,10] is what the cutter may choose)")
from scipy.cluster.hierarchy import fcluster

for k in range(2, 12):
    labels = fcluster(Z, t=k, criterion="maxclust")
    n_found = len(set(int(x) for x in labels))
    if n_found < 2:
        continue
    score = float(calinski_harabasz_score(matrix, labels))
    sizes = sorted(Counter(int(x) for x in labels).values(), reverse=True)
    marker = "  <-- shipped" if k == cut["top_partition_count"] else ""
    print(f"  k={k:2d}  CH={score:10.1f}  sizes={sizes}{marker}")


semantic shipped dynamic top_partition_count 3



silhouette              unconstrained k= 3  windowed 3..10 k= 3


calinski_harabasz       unconstrained k= 3  windowed 3..10 k= 3


davies_bouldin          unconstrained k= 2  windowed 3..10 k= 3

CH score by k (window in [3,10] is what the cutter may choose)


  k= 2  CH=       1.2  sizes=[278, 1]
  k= 3  CH=      58.3  sizes=[204, 74, 1]  <-- shipped
  k= 4  CH=      41.3  sizes=[199, 74, 5, 1]
  k= 5  CH=      32.4  sizes=[199, 71, 5, 3, 1]
  k= 6  CH=      26.7  sizes=[193, 71, 6, 5, 3, 1]
  k= 7  CH=      22.7  sizes=[189, 71, 6, 5, 4, 3, 1]
  k= 8  CH=      19.6  sizes=[189, 71, 5, 5, 4, 3, 1, 1]
  k= 9  CH=      23.9  sizes=[105, 84, 71, 5, 5, 4, 3, 1, 1]
  k=10  CH=      25.2  sizes=[105, 71, 42, 42, 5, 5, 4, 3, 1, 1]
  k=11  CH=      23.9  sizes=[93, 71, 42, 42, 12, 5, 5, 4, 3, 1, 1]


## 11. Diagnostic — do the two trees disagree enough to keep both?

The HTML switcher is only useful if Semantic and CODE put the same post in
**different neighborhoods**. Adjusted Rand Index (ARI) on the top partitions:

- ARI near 1 → switching trees barely changes the top folders; prefer the
  better-labeled tree and spend effort on **subcuts**.
- ARI nearer 0 → keep both; they are the two axes the mockup was built for.

Also compare LLM labels: if CODE internals are titled like topic clusters
(“Second Brain AI Integration”) rather than CODE stages, the labeler (which
only sees post **titles**) is describing MiniLM’s 0.30 slice, not Capture /
Organize / Distill / Express.


In [14]:
def top_labels(tree, cut_payload: dict) -> list[str]:
    assigned = {}
    for nid in cut_payload["top_partition_node_ids"]:
        node = tree.node(nid)
        for item_id in node["descendant_item_ids"]:
            assigned[item_id] = nid
    return [assigned[item_id] for item_id in tree.item_ids]


sem_labels = top_labels(sem, cuts["semantic"])
code_labels = top_labels(code_tree, cuts["CODE"])
print("ARI(semantic top, CODE top) =", round(adjusted_rand_score(sem_labels, code_labels), 3))

sem_of = {item: lab for item, lab in zip(sem.item_ids, sem_labels)}
code_of = {item: lab for item, lab in zip(code_tree.item_ids, code_labels)}

print()
print("contingency (semantic label × CODE label), counts")
sem_names = {row["id"]: f"{row['label'][:28]} n={row['n']}" for row in partition_rows(sem, cuts["semantic"])}
code_names = {row["id"]: f"{row['label'][:28]} n={row['n']}" for row in partition_rows(code_tree, cuts["CODE"])}
pairs = Counter((sem_of[i], code_of[i]) for i in sem.item_ids)
for (s, c), n in pairs.most_common():
    print(f"  {n:3d}  sem:{sem_names[s]:40}  code:{code_names[c]}")


ARI(semantic top, CODE top) = 0.713

contingency (semantic label × CODE label), counts
  188  sem:Second Brain AI Integration n=204         code:Second Brain AI Integration n=193
   69  sem:Second Brain Foundations n=74             code:Second Brain Implementation  n=85
   16  sem:Second Brain AI Integration n=204         code:Second Brain Implementation  n=85
    5  sem:Second Brain Foundations n=74             code:Second Brain AI Integration n=193
    1  sem:Second Brain Case Study: Sle n=1          code:Second Brain Case Study: Sle n=1


## 12. Diagnostic — which cuts to prioritize

A cut node is worth attention when:

1. **`max_depth` warning** — the cutter wanted to split further and could not.
   These are the largest “opaque folders” in the HTML (many posts hanging as
   direct leaves).
2. **Size imbalance** — a 200-post top folder is the real browse surface; the
   k=3 badge overstates how partitioned the corpus is.
3. **Repeated LLM labels** — sibling clusters with the same 2–6 word phrase
   are either over-split or under-named. Relabel or recut before treating
   them as distinct facets.
4. **High `hang` (direct leaves) at shallow depth** — those posts are visible
   next to big child folders; they behave like “uncategorized in this branch”.

Below: inventory of *this* tree’s shipped cut, then a priority table.


In [15]:
def flatten_cut(node: dict, tree, *, depth: int = 0, acc: list | None = None) -> list[dict]:
    rows = acc if acc is not None else []
    cid = node["canonical_node_id"]
    src = tree.node(cid)
    hang_ids = list(node.get("direct_leaf_node_ids") or [])
    child_nodes = list(node.get("children") or [])
    label = tree.label_for(cid) or default_node_label(tree, cid)
    rows.append(
        {
            "depth": depth,
            "id": cid,
            "kind": src["kind"],
            "n": src["descendant_count"],
            "label": label,
            "n_children": len(child_nodes),
            "n_hang": len(hang_ids),
            "hang_share": (len(hang_ids) / src["descendant_count"]) if src["descendant_count"] else 0.0,
        }
    )
    for child in child_nodes:
        flatten_cut(child, tree, depth=depth + 1, acc=rows)
    return rows


def warning_ids(cut_payload: dict) -> set[str]:
    found = set()
    for warning in cut_payload.get("warnings") or []:
        token = warning.split()[0]
        if token.startswith("member_"):
            found.add(token)
    return found


inventory = flatten_cut(cut["root"], tree)
warned = warning_ids(cut)
label_counts = Counter(row["label"] for row in inventory if row["kind"] != "leaf" and row["depth"] > 0)

print(TREE_ID, "cut-visible nodes", len(inventory), "warnings", len(warned))
print()
print("repeated labels (count > 1 among visible internals)")
for label, n in label_counts.most_common():
    if n < 2:
        continue
    sizes = [row["n"] for row in inventory if row["label"] == label]
    print(f"  {n}×  {label!r}  sizes={sizes}")

print()
print("priority: warned lumps, then large hang, then large shallow nodes")
ranked = sorted(
    [row for row in inventory if row["kind"] != "leaf" and row["depth"] > 0],
    key=lambda r: (
        0 if r["id"] in warned else 1,
        -r["n_hang"],
        -r["n"],
        r["depth"],
    ),
)
print(f"{'pri':>3} {'d':>2} {'n':>4} {'hang':>4} {'kids':>4}  {'warn':<5}  label")
for i, row in enumerate(ranked[:20], start=1):
    flag = "WARN" if row["id"] in warned else ""
    print(
        f"{i:3d} {row['depth']:2d} {row['n']:4d} {row['n_hang']:4d} {row['n_children']:4d}  "
        f"{flag:<5}  {row['label'][:50]}"
    )


semantic cut-visible nodes 65 warnings 4

repeated labels (count > 1 among visible internals)
  7×  'Second Brain Foundations'  sizes=[8, 3, 74, 45, 28, 12, 6]
  3×  'Building a Second Brain Updates'  sizes=[42, 32, 6]
  3×  'Digital Note-Taking Strategies'  sizes=[4, 42, 2]
  2×  'Book Publishing and Promotion'  sizes=[43, 34]
  2×  'Digital Organization Frameworks'  sizes=[50, 3]
  2×  'Personal Productivity Frameworks'  sizes=[37, 30]

priority: warned lumps, then large hang, then large shallow nodes
pri  d    n hang kids  warn   label
  1  5   30   30    0  WARN   Personal Productivity Frameworks
  2  5   18   18    0  WARN   Book Publishing and Marketing
  3  5   12   12    0  WARN   Second Brain Methodology
  4  5   12   12    0  WARN   Second Brain Foundations
  5  3   10   10    0         Online Course Creation Strategies
  6  4   10   10    0         Second Brain App Selection
  7  4    8    8    0         Second Brain Foundations
  8  4    8    8    0         Second Brain Eve

### Sample titles inside a prioritized node

Set `FOCUS` to a label from the table (or a `member_…` id). This is the
fastest way to decide whether a lump should be recut, relabeled, or left as
a “misc / updates” bucket.


In [16]:
FOCUS = ranked[0]["label"] if ranked else tree.label_for(tree.root_node_id)

focus_rows = [row for row in ranked if row["label"] == FOCUS or row["id"] == FOCUS]
focus = focus_rows[0] if focus_rows else ranked[0]
node = tree.node(focus["id"])
print(f"{TREE_ID}  {focus['label']}  n={focus['n']}  depth={focus['depth']}  id={focus['id'][:22]}…")
print()
for item_id in node["descendant_item_ids"][:24]:
    doc = tree.documents_by_id[item_id]
    print(f"  {item_id[:48]:48}  {doc.title[:70]}")
if node["descendant_count"] > 24:
    print(f"  … {node['descendant_count'] - 24} more")


semantic  Personal Productivity Frameworks  n=30  depth=5  id=member_fa3ae54a7efaa0b…

  0064-project-people-vs-area-people-are-you-runni  Project People vs. Area People: Are You Running a Sprint Or a Marathon
  0086-divergence-and-convergence-the-two-fundamen  Divergence and Convergence: The Two Fundamental Stages of the Creative
  0094-how-to-build-your-personal-productivity-sta  How to Build Your Personal Productivity Stack
  0109-intermediate-packets-in-the-wild             Intermediate Packets in the Wild
  0117-mise-en-place-for-knowledge-workers-6-pract  Mise-en-Place for Knowledge Workers: 6 Practices for Working Clean
  0142-the-4-levels-of-personal-knowledge-manageme  The 4 Levels of Personal Knowledge Management
  0226-just-in-time-pm-21-workflow-strategies       Just-in-Time PM #21: Workflow Strategies
  0236-just-in-time-pm-20-speed-as-a-capability     Just-In-Time PM #20: Speed as a Capability
  0237-just-in-time-pm-19-explosive-inspiration     Just-In-Time PM #19: Explos

## 13. Labels — gpt-4.1-mini on cut-visible internals only

**Choice.** Temperature 0, model `gpt-4.1-mini-2025-04-14`, 2–6 word noun
phrase, titles only (no body). Leaves keep post titles. Roots are forced to
`"semantic"` / `"CODE"`. Raw responses are cached under
`artifacts/forte_blogs/labels/raw/{tree_id}/`.

New linkage ⇒ new `member_*` ids, so a previous TF-IDF packet’s cache will
not hit. If `OPENAI_API_KEY` is missing the export still writes trees/cuts
but the HTML adapter **requires** labels.

The labeler never sees CODE class names. Combined with MiniLM’s 0.30 weight
on the CODE tree, that is why CODE folders read as topics.


In [17]:
from export_forte_orchard import cut_internal_ids, LABEL_MODEL, LABEL_SET_NAME

internal_ids = cut_internal_ids(cut["root"])
mapping = tree.labels[LABEL_SET_NAME]
print("label set", LABEL_SET_NAME, "model", LABEL_MODEL)
print("cut-visible internals", len(internal_ids))
print("labels on those ids  ", sum(1 for nid in internal_ids if mapping.get(nid)))
print("root label           ", mapping.get(tree.root_node_id))

leaf_ids = [nid for nid, node in tree.nodes.items() if node["kind"] == "leaf"]
leaf_is_title = 0
for nid in leaf_ids:
    item_id = tree.nodes[nid]["item_id"]
    doc = tree.documents_by_id[item_id]
    if mapping.get(nid) == (doc.title if doc.title.strip() else item_id):
        leaf_is_title += 1
print("leaves labeled with post title", leaf_is_title, "/", len(leaf_ids))


label set gpt41mini model gpt-4.1-mini-2025-04-14
cut-visible internals 65
labels on those ids   65
root label            semantic
leaves labeled with post title 279 / 279


## 14. Adapt to `visual_search_forte.html`

**Choice.** The live mockup was built for AppWorld’s `{domain, function}`
slots. Forte does not invent a new JS contract. It **reuses the slots**:

| Orchard tree | HTML slot | Button / sidebar |
|---|---|---|
| semantic | `TREES.domain` / `doc.domCluster` | Semantic |
| CODE | `TREES.function` / `doc.fnCluster` | CODE |

Adapter rules (`scripts/adapt_forte_to_mockup.py`):

- View ids are `d/…` or `f/…` plus 8-char membership-hash slugs.
- Root becomes `{id: "root", label: "All Documents", count: 279}`.
- Cut internals become labeled folders; cut leaves become posts (`count: 1`).
- `direct_leaf_node_ids` are promoted into `children` **for the mockup only**
  (the cut JSON on disk still separates them).
- Live badge `279 docs · N top-level clusters` is rewritten from the semantic
  cut’s `top_partition_count`.
- `file://` cannot `fetch()` JSON, so data is a sibling `<script src>` of
  `mockup_data.js`. Standalone copies local `d3.min.js` from the AppWorld packet.

Sanity the adapter enforces: 279 docs, every doc has both cluster ids, every
adapted leaf is referenced, root child counts match the live cuts.


In [18]:
html = (VIEW / "visual_search_forte.html").read_text(encoding="utf-8")
js_path = PACKET / "view" / "mockup_data.js"
print("html exists     ", (VIEW / "visual_search_forte.html").is_file())
print("mockup_data.js  ", js_path.is_file(), "bytes", js_path.stat().st_size if js_path.is_file() else 0)
print("script src      ", 'src="artifacts/forte_blogs/view/mockup_data.js"' in html)
print("semantic button ", 'switchTree(\'domain\')">Semantic' in html or 'switchTree("domain")">Semantic' in html)
print("CODE button     ", 'switchTree(\'function\')">CODE' in html or 'switchTree("function")">CODE' in html)
print("badge mentions 3", "279 docs · 3 top-level clusters" in html)
print("fusion caption  ", "MiniLM 0.66 / TF-IDF 0.34" in html)

standalone = sorted((PACKET / "standalone").glob("orchard_view_forte_*/visual_search_forte.html"))
print("standalone html ", standalone[-1].relative_to(VIEW) if standalone else None)

# Confirm every orchard doc made it into the JS payload without executing JS.
text = js_path.read_text(encoding="utf-8")
missing = [doc.item_id for doc in orchard.documents if doc.item_id not in text]
print("docs missing from mockup_data.js", len(missing), missing[:3])
print("semantic top children (HTML domain slot) should be", cuts["semantic"]["top_partition_count"])
print("CODE top children     (HTML function slot) should be", cuts["CODE"]["top_partition_count"])


html exists      True
mockup_data.js   True bytes 2788324
script src       True
semantic button  True
CODE button      True
badge mentions 3 True
fusion caption   True
standalone html  artifacts\forte_blogs\standalone\orchard_view_forte_202616081235\visual_search_forte.html


docs missing from mockup_data.js 0 []
semantic top children (HTML domain slot) should be 3
CODE top children     (HTML function slot) should be 3


## 15. How to choose from here

**Keep both trees** if section 11’s ARI is clearly below ~0.8 *and* the CODE
folders add a process-stage reading you cannot get from Semantic. If CODE
folders are topic-labeled and the contingency table is nearly diagonal, the
second tree is a mildly reweighted semantic tree — either raise `CODE_raw_js`
toward 1.0 and rebuild, or treat CODE as secondary.

**Prioritize recuts** in this order:

1. Nodes in the warning list (depth cap hid a split the optimizer liked).
2. The giant top folder (~200 posts). Almost all browse UX lives there; the
   k=3 top cut is mostly “big pile / smaller pile / outlier”.
3. Repeated labels on siblings — relabel cheaply before rebuilding.
4. The shared singleton outlier — it will stay a top-level tile on both
   trees until you special-case it or change fusion.

**Do not** recut on raw TF-IDF or raw CODE rows. Rebuild fused D from
`orchard.layer_matrices` as in section 6, then `build_dynamic_cut`.

Optional: bump `max_depth` above 5, or widen `max_width`, and compare to the
shipped JSON with the rebuild cell below.


## 16. Optional — rebuild a cut on this tree (in memory)

Does not write artifacts. Change `max_depth` / `max_width` to see what the
HTML *would* show. Validate against the same tree object.


In [19]:
rebuilt = build_dynamic_cut(
    tree,
    top_criterion="optimal",
    cut_optimizer="calinski_harabasz_score",
    feature_matrix=matrix,
    cut_polarity=1,
    min_width=3,
    max_width=10,
    target_width=None,
    max_depth=5,
    threshold_steps=32,
)
validate_dynamic_cut(rebuilt, tree)
print("shipped k", cut["top_partition_count"], "rebuilt k", rebuilt["top_partition_count"])
print("shipped warnings", len(cut.get("warnings") or []), "rebuilt warnings", len(rebuilt.get("warnings") or []))
print()
walk_cut(rebuilt["root"], tree, max_depth=1, show_leaf=False, show_kind=False)


shipped k

 3 rebuilt k 3
shipped warnings 4 rebuilt warnings 4

semantic  (279)  hang=1
  Second Brain AI Integration  (204)  hang=1
  Second Brain Foundations  (74)


## 17. Rebuild the HTML packet (do not run from this notebook)

Full neural rebuild (MiniLM + ModernBERT + OpenAI labels):

```bash
uv run python scripts/export_forte_orchard.py
uv run python scripts/adapt_forte_to_mockup.py
```

Then open `visual_search_forte.html` or the stamped folder under
`artifacts/forte_blogs/standalone/`.

Orchard source stays import-only. AppWorld artifacts/scripts stay untouched.
The CODE YAML seed is the only file under `artifacts/forte_blogs/` the export
script will not delete.
